In [15]:
from dotenv import load_dotenv

load_dotenv()

True

In [16]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

In [17]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house via an image.

Create a list of all the ingredients from the image and confirm with the user by returning the list as text output.

Once the user confirms, use the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [18]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import InMemorySaver
import os
load_dotenv()

model = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
)

agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [19]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [20]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [21]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Here is the list of ingredients left in my refridgerator"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

# response = agent.invoke(
#     {"messages": [HumanMessage(content="I have some leftover potato and rice. What can I make?")]},
#     config
# )

response = agent.invoke(
    {"messages": [multimodal_question]}, config
)

print(response['messages'][-1].content)

[{'type': 'text', 'text': "It looks like you have a great selection of fresh vegetables! Based on the image, here are the ingredients I've identified:\n\n*   **Tomatoes** (on the vine)\n*   **Green Cabbage** (Savoy or similar)\n*   **Carrots**\n*   **Red Onion**\n*   **Beets**\n*   **Green Onions** (Scallions)\n*   **Green Leaf Lettuce**\n*   **Yellow Bell Pepper**\n*   **Red Bell Pepper**\n*   **Purple Leaf Lettuce or Radicchio**\n\nPlease confirm if this list is correct, or let me know if I missed anything or if you have any pantry staples (like oil, vinegar, grains, or proteins) you'd like me to include in the recipe search!", 'extras': {'signature': 'EuUHCuIHAWkUfROoP/R2Mvx/Yp8U00mPMkwkaF7Wxx6V0xhY0abK3yyQHTlb+qu363ymr8dxAmjqovjixfKVp7l+75Id6dpWG0uH/u/jx/O7NAyWWNprF/lN5E2xjSc0IdddPXYhXuo+dT+AMt+hvSSy7UV+4OfqcKULBCU9DLlXr16COOwRA10tYK0RegD4WMwF4j0nGNmynB5pZejcwN4CttkdXf7BtTmQN8DCYNHA8neiqCyFS6zIZqwB/bFoaePpLGP/hCOjAibp+3Kh24AEZIF/V/8rKizXMpWQkw6nULA/zaNNOfoy6vCL5ScSbKuUivaVapchhuU5r

In [22]:
question = HumanMessage(content="I confirm that list is correct. What can i make for tonight's dinner?")

response = agent.invoke(
    {"messages": [question]},
    config,  
)

print(response['messages'][-1].content)

[{'type': 'text', 'text': 'Since you have such a colorful variety of fresh produce, you have some great options for a healthy and vibrant dinner! Here are three different ways we could go:\n\n### 1. **Hearty Vegetarian "Borscht-Style" Soup**\nThis is the perfect way to use the **beets, cabbage, carrots, onions, and tomatoes**. It’s warm, filling, and incredibly nutritious.\n*   **Vibe:** Cozy, rustic, and great for leftovers.\n*   **Key Ingredients used:** Beets, cabbage, carrots, red onion, tomatoes.\n\n### 2. **Sheet-Pan Roasted Root Veggies & Cabbage "Steaks"**\nRoasting **beets and carrots** brings out their natural sweetness. You can slice the **cabbage** into thick "steaks" and roast them alongside the **peppers and onions** for a caramelized, smoky dinner.\n*   **Vibe:** Low'}, {'type': 'text', 'text': '-effort (one pan!), crispy textures, and naturally sweet.\n*   **Key Ingredients used:** Cabbage, beets, carrots, red & yellow peppers, red onion.\n\n### 3. **Rainbow Veggie Stir